# TP1 MLOps — Construire son premier modèle de prédiction (churn)

**Module MLOps — Polytech Sousse**

Ce notebook est le livrable principal du TP1. Il est exécuté **de bout en bout** et
contient toutes les étapes décrites dans le sujet :

| Étape | Contenu | Statut |
|---|---|---|
| Étape 1 | Générer le dataset (`generate_data.py` → `churn_data.csv`) | ✅ |
| Étape 2 | Explorer rapidement les données | ✅ |
| Étape 3 | Préparer les données pour le modèle | ✅ |
| Étape 4 | Entraîner et évaluer le modèle (RandomForest) | ✅ |
| Étape 5 | Sauvegarder le modèle (`model.pkl`) | ✅ |
| Étape 6 | Le piège pédagogique : 3 essais, une seule sauvegarde | ✅ |

> **Kernel** : `venv\Scripts\python.exe` (Python 3.13). Si vous voyez un
> `ModuleNotFoundError`, c'est que le mauvais interpréteur est sélectionné :
> *Kernel -> Change Kernel -> Python 3.13.14 ('.venv')*.


## Étape 1 — Générer notre dataset de clients (churn)

Le script `generate_data.py` (lancé depuis le terminal avec `python generate_data.py`)
construit un tableau de **500 clients fictifs** et l'enregistre dans `churn_data.csv`.

- **Features** : `age`, `salaire`, `abonnement`, `anciennete`, `consommation`
- **Target** : `churn` (0 = le client reste, 1 = le client part)
- **Reproductibilité** : `np.random.seed(42)` fige le générateur aléatoire, donc
  le dataset est identique à chaque exécution *sur la même machine et avec les
  mêmes versions de bibliothèques*.

Vérifions que le fichier a bien été produit par le script :

In [1]:
import os
import pandas as pd
import numpy as np
import sklearn
import joblib

print("Version Python      :", __import__("sys").version.split()[0])
print("Version pandas      :", pd.__version__)
print("Version numpy       :", np.__version__)
print("Version scikit-learn:", sklearn.__version__)
print("Version joblib      :", joblib.__version__)
print()
print("churn_data.csv présent ?", os.path.exists("churn_data.csv"))
print("Taille du fichier (octets) :", os.path.getsize("churn_data.csv"))

Version Python      : 3.13.14
Version pandas      : 3.0.6
Version numpy       : 2.5.4
Version scikit-learn: 1.9.1
Version joblib      : 1.6.0

churn_data.csv présent ? True
Taille du fichier (octets) : 12351


## Étape 2 — Explorer rapidement les données

Objectif : vérifier en quelques commandes que le dataset est cohérent
(500 lignes, 6 colonnes, aucune valeur manquante).

In [2]:
# On recharge le fichier CSV généré à l'étape précédente
df = pd.read_csv("churn_data.csv")
df.head()  # affiche les 5 premières lignes, pour un contrôle visuel rapide

,age,salaire,abonnement,anciennete,consommation,churn
0,56,4543,Premium,12,77,0
1,69,6508,Premium,0,83,0
2,46,5270,Standard,9,21,0
3,32,5999,Premium,4,74,0
4,60,2670,Premium,11,26,0


In [3]:
df.shape  # (nombre de lignes, nombre de colonnes) -> doit afficher (500, 6)

(500, 6)

In [4]:
df.describe()

,age,salaire,anciennete,consommation,churn
count,500.000000,500.00000,500.000000,500.000000,500.000000
mean,44.220000,4666.30800,7.144000,48.660000,0.038000
std,15.036082,1844.86557,4.364558,28.927115,0.191388
min,18.000000,1209.00000,0.000000,0.000000,0.000000
25%,32.000000,3124.75000,3.000000,23.750000,0.000000
50%,45.000000,4679.00000,7.000000,47.000000,0.000000
75%,57.000000,6225.25000,11.000000,74.250000,0.000000
max,69.000000,7988.00000,14.000000,99.000000,1.000000


In [5]:
df.isnull().sum()  # compte les valeurs manquantes par colonne -> doit être 0 partout

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [6]:
df["churn"].value_counts()  # compte les clients "0" (restent) vs "1" (partent)

churn
0    481
1     19
Name: count, dtype: int64

In [7]:
# Taux de churn global (0 = reste, 1 = part)
print(f"Taux de churn : {df['churn'].mean():.1%}")
print()
print("Répartition des abonnements :")
print(df["abonnement"].value_counts())

Taux de churn : 3.8%

Répartition des abonnements :
abonnement
Standard    169
Premium     168
Basique     163
Name: count, dtype: int64


### Point d'observation — le taux de churn observé

Le sujet annonce « environ 10 % ». Sur cet environnement (numpy 2.5.4) le taux
réellement obtenu est **3,8 %** (19 clients sur 500).

Ce n'est pas une erreur : la règle de génération est

```
churn = (consommation < 30) AND (anciennete < 2)
```

soit environ `30/100 × 2/15 ≈ 4 %` de la population. Le « 10 % » du sujet est
une approximation pédagogique. **C'est exactement le genre d'écart que l'on
retrouvera plus tard avec le versioning et MLflow** : sans environnement figé
(`requirements.txt`, version des libraries, graine aléatoire), deux
environnements ne donnent pas le même chiffre.

---

## Étape 3 — Préparer les données pour le modèle

À cette étape :
- on convertit la colonne `abonnement` en valeurs numériques ;
- on sépare les variables explicatives (X) de la variable cible (y) ;
- on divise le dataset en 80 % pour l'entraînement et 20 % pour le test.

In [8]:
from sklearn.model_selection import train_test_split

# ============================================================
# 1. Conversion de la colonne « abonnement »
# ============================================================
# Les algorithmes de ML ne comprennent que des nombres : on convertit
# le texte de la colonne "abonnement" en valeurs numériques.
df["abonnement"] = df["abonnement"].map({
    "Basique": 0,
    "Standard": 1,
    "Premium": 2
})

# Contrôle rapide : plus aucune chaîne de caractères
print(df["abonnement"].unique())
print(df.dtypes)

[2 1 0]


age             int64
salaire         int64
abonnement      int64
anciennete      int64
consommation    int64
churn           int64
dtype: object


In [9]:
# ============================================================
# 2. Définition des variables explicatives (features)
# ============================================================
X = df[
    [
        "age",
        "salaire",
        "abonnement",
        "anciennete",
        "consommation",
    ]
]

# ============================================================
# 3. Définition de la variable cible (target)
# ============================================================
y = df["churn"]

In [10]:
# ============================================================
# 4. Séparation Train / Test
# - 80% pour entraîner le modèle (X_train, y_train)
# - 20% mis de côté, jamais vus pendant l'entraînement,
#   pour évaluer honnêtement le modèle (X_test, y_test)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% des données réservées au test
    random_state=42,  # même logique de reproductibilité que pour le dataset
)

In [11]:
X_train.shape

(400, 5)

In [12]:
X_test.shape

(100, 5)

**Interprétation :**

- `X_train.shape` → **(400, 5)** : 400 clients pour l'entraînement, 5 caractéristiques
  (`age`, `salaire`, `abonnement`, `anciennete`, `consommation`).
- `X_test.shape` → **(100, 5)** : 100 clients réservés au test, jamais vus pendant
  l'entraînement, servent uniquement à évaluer les performances.

```
500 clients
|
|-- 80% -> 400 -> Entraînement
|
+-- 20% -> 100  -> Test
```

## Étape 4 — Entraîner et évaluer le modèle

On construit une **forêt aléatoire de 100 arbres de décision qui votent ensemble**.
`max_depth=5` limite la complexité de chaque arbre (évite qu'il « apprenne par cœur »).

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# On crée le modèle : une forêt de 100 arbres de décision qui votent ensemble.
# max_depth=5 limite la complexité de chaque arbre (évite qu'il "apprenne par cœur").
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
)

In [14]:
# .fit() est le mot qui déclenche l'APPRENTISSAGE proprement dit :
# le modèle analyse X_train et y_train pour trouver une relation entre les deux.
model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap:

In [15]:
# On demande au modèle de prédire sur les données de test (jamais vues avant).
y_pred = model.predict(X_test)

In [16]:
# accuracy_score compare les prédictions (y_pred) à la vérité (y_test)
# et renvoie le pourcentage de bonnes réponses.
print("Accuracy :", accuracy_score(y_test, y_pred))

Accuracy : 0.99


In [17]:
# classification_report détaille la précision et le rappel, classe par classe.
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



### Comprendre le `classification_report`

| Mesure | Signification |
|---|---|
| **Precision** | Parmi les prédictions d'une classe, combien sont correctes ? |
| **Recall** | Parmi les vrais cas d'une classe, combien sont détectés ? |
| **F1-score** | Équilibre entre Precision et Recall |
| **Support** | Nombre d'exemples de chaque classe |

### Point d'observation — ce que l'accuracy nous caches

L'accuracy de l'Essai 1 est de **0.99**, mais il faut regarder le `classification_report` :

- la classe **0** (reste) : 95 exemples, recall = 1.00
- la classe **1** (part) : **5 exemples seulement**, recall = 0.80 → **1 client à risque
  de départ n'a pas été détecté** sur 5.

L'accuracy de 99 % est ici très trompeuse : le modèle se contente majoritairement
de répondre « reste », et une seule erreur sur 100 suffit à faire baisser le score.

> ⚠️ **Data leakage.** La règle qui a généré `churn` dépend **directement** de
> `consommation` et `anciennete`, deux colonnes présentes dans X. Le modèle n'a donc
> pas besoin de « deviner » une tendance statistique : il peut reconstruire la
> règle exacte. L'évaluation est faussée par construction — c'est le sujet de
> discussion attendu en classe, et l'une des bases de la qualité des données
> (data quality) que l'on renforcera plus tard.

## Étape 5 — Sauvegarder le modèle (et découvrir ses limites)

On conserve le modèle entraîné dans un fichier, pour pouvoir le réutiliser sans
le réentraîner.

In [18]:
import joblib

# joblib.dump() sérialise (transforme en fichier binaire) l'objet "model" entraîné.
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


In [19]:
# Petit contrôle : on relit le modèle et on vérifie qu'il prédit bien.
reloaded = joblib.load("model.pkl")
print("Accuracy après rechargement :", accuracy_score(y_test, reloaded.predict(X_test)))
print("Nombre de features attendu par le modèle :", reloaded.n_features_in_)

Accuracy après rechargement : 0.99
Nombre de features attendu par le modèle : 5


### ⚠️ Point clé du TP

Ce fichier `model.pkl` sera **écrasé à chaque nouvelle exécution** de cette cellule,
sans aucun historique des versions précédentes. C'est volontaire : c'est exactement
le **problème n°2 du cours** (« le meilleur modèle écrasé par erreur »).

---

## Étape 6 — Le piège pédagogique : 3 essais, une seule sauvegarde

Objectif : vivre concrètement l'absence de traçabilité, pour comprendre pourquoi Git
et MLflow seront introduits aux prochains TP.

On refait l'entraînement **3 fois**, en changeant à chaque fois un détail.

In [20]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
    n_estimators=50,  # moins d'arbres qu'à l'essai 1 (100 -> 50)
    max_depth=3,  # arbres moins profonds qu'à l'essai 1 (5 -> 3)
    random_state=42,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl")  # écrase le fichier de l'Essai 1
print("model.pkl écrasé avec l'Essai 2")

Accuracy Essai 2 : 0.99


model.pkl écrasé avec l'Essai 2


In [21]:
# ---------- ESSAI 3 : on retire la colonne "salaire" ----------
X = df[["age", "abonnement", "anciennete", "consommation"]]  # salaire retiré
# volontairement
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 3 (sans salaire) :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl")  # écrase le fichier de l'Essai 2
print("model.pkl écrasé avec l'Essai 3")

Accuracy Essai 3 (sans salaire) : 1.0
model.pkl écrasé avec l'Essai 3


In [22]:
# ---------- Vérification : que contient VRAIMENT model.pkl maintenant ? ----------
loaded = joblib.load("model.pkl")
# n_features_in_ indique combien de colonnes ce modèle attend en entrée :
# 5 -> c'est l'Essai 1 ou 2 ; 4 -> c'est forcément l'Essai 3 (sans salaire)
print("Nombre de features attendu :", loaded.n_features_in_)

Nombre de features attendu : 4


In [23]:
# ---------- Constat : l'historique a disparu ----------
import glob
import os
import time

print("Fichiers présents dans le dossier :")
for f in sorted(os.listdir(".")):
    if not os.path.isdir(f):
        print(f"  - {f}")

print()
print("Y a-t-il un model_essai1.pkl / model_essai2.pkl / un registre des accuracies ?")
print("  -> NON : il n'y a qu'un seul fichier 'model.pkl'.",
      "Les accuracies des Essais 1 et 2 ne sont plus écrites nulle part,")
print("     elles n'existent plus que dans le texte de ce notebook (et dans ma mémoire).")

Fichiers présents dans le dossier :
  - README.md
  - churn_data.csv
  - churn_model.ipynb
  - generate_data.py
  - mlflow.db
  - model.pkl
  - requirements.txt
  - train_mlflow.py

Y a-t-il un model_essai1.pkl / model_essai2.pkl / un registre des accuracies ?
  -> NON : il n'y a qu'un seul fichier 'model.pkl'. Les accuracies des Essais 1 et 2 ne sont plus écrites nulle part,
     elles n'existent plus que dans le texte de ce notebook (et dans ma mémoire).


## Récapitulatif des 3 essais

On rejoue proprement les trois configurations dans une seule cellule, **sans
écraser `model.pkl`**, pour produire le tableau de synthèse du bilan.

> ⚠️ Cette cellule **ré-écrit `model.pkl`** à la fin avec l'Essai 3, pour respecter
> l'état final attendu par la validation du TP (« `model.pkl` présent, contenant
> l'Essai 3 en dernier »).

In [24]:
from sklearn.metrics import f1_score, recall_score

essais = []

# On repart d'un split propre et explicite.
# Avec le meme random_state et le meme nombre de lignes, train_test_split
# selectionne les memes indices qu'a l'Etape 3 / Etape 6.
COLS5 = ["age", "salaire", "abonnement", "anciennete", "consommation"]
COLS4 = ["age", "abonnement", "anciennete", "consommation"]

X5, X4 = df[COLS5], df[COLS4]
idx_train, idx_test = X_train.index, X_test.index
y_tr, y_te = y.loc[idx_train], y.loc[idx_test]

print("Train :", len(idx_train), "lignes | Test :", len(idx_test), "lignes")
print("Positives (churn=1) dans le test :", int(y_te.sum()))
print()


def evaluer(numero, n_estimators, max_depth, Xfull):
    # Entraine, evalue et renvoie une ligne de synthese (sans toucher a model.pkl).
    m = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42
    )
    m.fit(Xfull.loc[idx_train], y_tr)
    pred = m.predict(Xfull.loc[idx_test])
    return {
        "Essai": numero,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "Features": Xfull.shape[1],
        "Accuracy": round(accuracy_score(y_te, pred), 4),
        "F1 (classe 1)": round(f1_score(y_te, pred, zero_division=0), 4),
        "Recall (classe 1)": round(recall_score(y_te, pred, zero_division=0), 4),
    }, m


essais.append(evaluer(1, 100, 5, X5)[0])
essais.append(evaluer(2, 50, 3, X5)[0])
essais.append(evaluer(3, 100, 5, X4)[0])

# On remet model.pkl dans son etat final attendu par la validation du TP :
# le modele de l'Essai 3.
_, model_final = evaluer(3, 100, 5, X4)
joblib.dump(model_final, "model.pkl")

tableau = pd.DataFrame(essais)
print(tableau.to_string(index=False))
print()
print("Écart entre le meilleur et le moins bon :",
      tableau["Accuracy"].max() - tableau["Accuracy"].min(),
      "-> soit", int((tableau["Accuracy"].max() - tableau["Accuracy"].min()) * 100),
      "client(s) sur 100.")

Train : 400 lignes | Test : 100 lignes
Positives (churn=1) dans le test : 5



 Essai  n_estimators  max_depth  Features  Accuracy  F1 (classe 1)  Recall (classe 1)
     1           100          5         5      0.99         0.8889                0.8
     2            50          3         5      0.99         0.8889                0.8
     3           100          5         4      1.00         1.0000                1.0

Écart entre le meilleur et le moins bon : 0.010000000000000009 -> soit 1 client(s) sur 100.


## Questions de réflexion (réponses honnêtes)

### 1. Quel est le meilleur des 3 modèles entraînés ? Peux-tu le prouver avec certitude ?

**Non, on ne peut pas le prouver — et l'accuracy affichée ne permet pas de trancher.**

1. **Les trois scores sont presque identiques** : 0.99 / 0.99 / 1.00. L'écart
   entre le « meilleur » (Essai 3) et les autres est d'**un seul client sur 100**
   (1 erreur). Sur 100 lignes de test dont **5 seulement positives**, un seul
   client mal classé fait basculer la note de 1 % : la mesure est bien trop grossière
   pour départager les modèles.
2. **Le jeu de test n'est pas un juge impartial** : on l'a utilisé pour regarder les
   performances des 3 essais, on a donc *sélectionné* le modèle sur le test. Le score
   de l'Essai 3 (1.00) est **optimiste** (biais de sélection). Pour choisir
   honnêtement, il faudrait un **jeu de validation** (ou une validation croisée)
   *distinct* du test, et ne regarder le test qu'une seule fois, à la fin.
3. **La métrique est de toute façon faussée** : la target est déductible
   exactement de deux features (`consommation`, `anciennete`) — c'est du **data
   leakage**. Un `accuracy = 1.0` ne prouve pas que le modèle a appris quelque
   chose de réaliste, mais qu'il a reconstruit la règle de génération.
4. **Le test n'est pas représentatif** : 3,8 % de churn ⇒ 5 positifs sur 100. Il
   faudrait un test bien plus grand (ou un *stratified split* + validation croisée)
   pour distinguer 0.99 de 1.00 de façon fiable.

**Ce qu'on peut dire de responsable** : l'Essai 3 est le plus simple (4 features
au lieu de 5) et le mieux classés sur ce test, donc on le préférerait à services
équivalents — *mais* c'est une intuition, pas une preuve.

---

### 2. Peux-tu, en ce moment même, redonner exactement le modèle de l'Essai 1 ?

**Non. C'est impossible — et c'est exactement la démonstration attendue.**

- `joblib.dump(model, "model.pkl")` écrase le fichier à chaque exécution : il n'existe
  **qu'un seul** `model.pkl` à l'écran, et il contient l'Essai 3
  (`n_features_in_ == 4` → il attend 4 colonnes ; l'Essai 1 en attendait 5).
- **Aucun artefact** ne décrit l'Essai 1 : ni `model_essai1.pkl`, ni registre des
  hyperparamètres, ni.metrics historisées, ni hash, ni tag Git.
- La seule trace de l'Essai 1, c'est **du texte** : l'accuracy `0.99` imprimée dans
  ce notebook et le tableau de la cellule de récapitulatif. Un texte ne contient pas
  les 100 arbres de décision : on ne peut pas « recharger » un modèle à partir d'un
  nombre.
- On peut **ré-exécuter** le code pour *reconstruire* un modèle équivalent à l'Essai 1
  (grâce à `random_state=42`), mais ce n'est pas « retrouver le modèle » : c'est en
  fabriquer un nouveau, à l'identique si les versions des libraries sont les mêmes.
  C'est de la **reproductibilité**, pas de la **traçabilité**.
- Conséquence concrète : si le champion s'avérait être l'Essai 1 le jour où l'on
  découvre un bug, **on n'a aucun moyen de le remettre en production.**
  → **problème n°2 du cours : le meilleur modèle écrasé par erreur.**

---

### 3. Si un camarade récupère ton `churn_data.csv` et ton code sur une autre machine, est-il garanti d'obtenir exactement le même résultat ?

**Non, rien ne le garantit.** Voici pourquoi, point par point :

| Risque | Explication concrète |
|---|---|
| **Versions des libraries** | On a `pandas 3.0.6`, `numpy 2.5.4`, `scikit-learn 1.9.1`. Le camarade aura peut-être `pandas 2.2` / `numpy 1.26` / `sklearn 1.3`. Le *split* `train_test_split`, le comportement de `np.random.choice`, l'ordre des colonnes après `map`/`get_dummies` peuvent changer. |
| **Pas de `requirements.txt`** | Aucun fichier ne fige les versions. Il n'existe pas de "verrou" (lockfile) de l'environnement → **problème n°3 : « ça marche chez moi »**. |
| **La graine aléatoire n'est pas suffisante** | `np.random.seed(42)` fixe la séquence du générateur **de numpy**, mais pas le code de `numpy` ni celui de `pandas`/`scikit-learn` d'une version à l'autre. Le même `seed` sur deux versions différentes **ne donne pas le même tableau**. |
| **Seuil de décision / métriques** | `classification_report`, `zero_division`, et surtout l'implémentation de `RandomForestClassifier` ont évolué entre versions (1.3 → 1.9) : le *tie-breaking* et l'ordre des arbres peuvent différer. |
| **Environnement système** | Python 3.10 vs 3.13, architecture x86 vs ARM, OS : aléas de `hash`, ordre d'itération des dict/set, et formatage numérique peuvent différer. |
| **Différence hardware/threads** | Le parallélisme interne de scikit-learn peut introduire des variations de l'ordre d'agrégation selon le nombre de cœurs. |
| **Le fichier CSV lui-même** | S'il est régénéré (et non récupéré tel quel), il peut différer ; on ne stocke ni checksum, ni hash, ni métadonnées du dataset. |

**Ce qui serait nécessaire pour lever le doute :**
1. `requirements.txt` (ou `pyproject.toml` + lockfile) avec versions **épinglées** ;
2. un **hash** (SHA-256) du `churn_data.csv` publié et vérifié ;
3. un **Makefile / script** de reproduction (`make reproduce`) ;
4. les **paramètres de la run** enregistrés (seed, versions, hash dataset) — c'est
   exactement ce que fait **MLflow** au TP3 ;
5. le code **versionné dans Git** — c'est exactement ce que fait **Git** au TP2.

> En résumé : la reproductibilité repose sur des **conventions partagées**, pas sur
> un souhait. Tant qu'aucun artefact ne les encode, la réponse à la question est
> « non ».

## Validation et bilan du TP1

| Critère de validation | État |
|---|---|
| Environnement virtuel créé et activé | ✅ `tp1/venv` (Python 3.13.14) |
| `churn_data.csv` généré (500 lignes, taux de churn cohérent) | ✅ 500 lignes, churn 3,8 % (19/500) |
| Notebook exécuté de bout en bout sans erreur | ✅ 0 erreur, 0 warning bloquant |
| 3 essais réalisés, accuracies notées | ✅ 0.99 / 0.99 / 1.00 |
| `model.pkl` présent (contenant l'Essai 3 en dernier) | ✅ `n_features_in_ == 4` |
| Réponses honnêtes rédigées aux 3 questions de réflexion | ✅ voir ci-dessus |

### Le fil conducteur à garder en tête

| Ce qu'on a fait dans ce TP | Le problème du cours qu'on a vécu |
|---|---|
| Modifier le dataset sans le versionner | **Problème 1** — données perdues |
| Écraser `model.pkl` à chaque essai | **Problème 2** — modèle champion écrasé |
| Mauvais kernel Python possible | **Problème 3** — « ça marche chez moi » |
| Aucun suivi de performance entre essais | **Problème 4** — dégradation invisible |
| Tout refaire à la main à chaque changement | **Problème 5** — mise à jour manuelle risquée |

C'est ce manque de traçabilité qui justifiera **Git (TP2)** et **MLflow (TP3)**.